# Audio-swap benchmark: does looking at the video actually help? (Kaggle GPU)

On AVID-Bench the contradictory clips have **edited** audio, so an audio-only model scores 0.96 without
looking at the video. This notebook removes that shortcut. It uses only AVID's **original, unedited**
clips:

* **consistent**: clip A's video with A's own audio
* **contradictory**: clip A's video with the audio of another clip B (same split, same length)

Every audio track and every video appears once in each class, so `audio_only` and `video_only` can
only reach chance (≈ 0.50). A model scores above that only by comparing audio with video. The swaps
are made from cached features, so no videos are re-encoded and there are no editing artefacts.

It trains all variants over 3 seeds on the swaps, and also tests the released AVID model
(`models/avid/`) on them.

**Before you run**
1. *Add Input* → your **`avid-subset`** dataset (same as the training notebook).
2. *Optional, saves ~1 hour:* *Add Input* → *Your Work* → **Notebooks** → the notebook of your earlier
   successful training run. Its saved CLIP + CLAP features are reused instead of being recomputed.
3. *Settings*: Accelerator **GPU T4 ×2**, Internet **On**.
4. *Save Version → Save & Run All*. Download `swap_results.zip` from the *Output* tab when it finishes.

In [ ]:
REPO_URL = "https://github.com/adhvaith267/Cross-Attention-Tensor-Fusion-for-Detecting-Audio-Visual-Semantic-Contradictions-in-Short-Form-Video.git"
BRANCH = "main"
SEEDS = "1 2 3"
# Run 1 used the default schedule (60 epochs, patience 15, lr 3e-4); the cross-attention models never left
# chance there. Run 2 gives every model the same longer, gentler schedule. Results go to reports/<EXPERIMENT>.
EXPERIMENT = "avid_swap_long"
SCHEDULE = "--set training.epochs=150 --set training.early_stopping_patience=50 --set training.lr=1e-4"

OUT = "/kaggle/working/output"   # kept after the session: features, checkpoints, reports
CODE = "/tmp/av_contradiction"   # code lives outside /kaggle/working (not saved)

## 1. Get the code

In [ ]:
import os
import subprocess

url, token = REPO_URL, None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass  # no secret attached: public repo
subprocess.run(["rm", "-rf", CODE], check=True)
r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, CODE], capture_output=True, text=True)
log = r.stdout + r.stderr
print(log.replace(token, "***") if token else log)
assert r.returncode == 0, "Clone failed: check REPO_URL, BRANCH and the GITHUB_TOKEN secret"
subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", REPO_URL], check=True)  # keep the token off disk
os.chdir(CODE)

## 2. Install and check the GPU

In [ ]:
!pip install -q -e ".[encoders]"
!python -m av_contradiction.cli.check_environment

## 3. Find the attached dataset

In [ ]:
import glob

MANIFEST = f"{CODE}/manifests/avid_subset.csv"
roots = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/test_1171_segments", recursive=True)})
if not roots:
    print("Nothing with test_1171_segments/ under /kaggle/input. What is attached:")
    for p in sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*"))[:30]:
        print("  ", p)
    raise SystemExit("Attach the AVID subset dataset: editor -> right panel -> Input -> + Add Input")
ROOT = roots[0]
n = len(glob.glob(f"{ROOT}/*_segments/*/*.mp4"))
print(f"dataset root: {ROOT} | {n} mp4 files")
assert n == 2600, f"expected 2600 videos, found {n}: is the upload complete?"

## 4. Reuse features from the earlier run (if attached)

Looks for the earlier notebook's feature cache under `/kaggle/input/` and copies the features of the
1,300 original-audio clips into this run. If it is not attached, step 5 extracts them (~1 hour).

In [ ]:
import shutil

HASH = subprocess.check_output(["python", "-c", "from av_contradiction.config import load_config; print(load_config().feature_hash())"],
                               text=True).strip()
CACHE = f"{OUT}/cache/{HASH}"
os.makedirs(CACHE, exist_ok=True)
prev = sorted(p for p in glob.glob(f"/kaggle/input/**/cache/{HASH}", recursive=True) if not p.startswith(OUT))
if prev:
    files = glob.glob(f"{prev[0]}/pos_*.pt")
    for f in files:
        shutil.copy2(f, CACHE)
    print(f"reused {len(files)} cached clips from {prev[0]}")
else:
    print("no earlier feature cache attached: step 5 will extract the 1,300 clips (~1 hour)")

## 5. Features of the original-audio clips (skipped where cached)

In [ ]:
import csv

CONSISTENT = f"{OUT}/avid_consistent.csv"
with open(MANIFEST, encoding="utf-8", newline="") as fh:
    reader = csv.DictReader(fh)
    keep = [r for r in reader if r["label"] == "consistent"]
    fields = reader.fieldnames
with open(CONSISTENT, "w", encoding="utf-8", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=fields)
    w.writeheader()
    w.writerows(keep)
print(len(keep), "original-audio clips")

PATHS = (f"--set data.root_dir={ROOT} --set data.feature_cache_dir={OUT}/cache "
         f"--set training.checkpoint_dir={OUT}/checkpoints --set evaluation.output_dir={OUT}/reports")
!python -m av_contradiction.cli.extract_features {PATHS} --set data.manifest={CONSISTENT} --set encoders.local_files_only=false

## 6. Build the swap manifests

* `avid_swap.csv`: partners drawn at random among clips of the same length (used for training and the main test)
* `avid_swap_hard.csv`: partners whose audio is most similar to the clip's own (e.g. speech replaced by other speech); used as a harder second test

Both print a balance check: every audio and every video once per class, partners from the same split.

In [ ]:
SWAP, HARD = f"{OUT}/avid_swap.csv", f"{OUT}/avid_swap_hard.csv"
!python -m av_contradiction.cli.make_swaps {PATHS} --source {CONSISTENT} --out {SWAP} --pairing random --seed 0
!python -m av_contradiction.cli.make_swaps {PATHS} --source {CONSISTENT} --out {HARD} --pairing hard
!python -m av_contradiction.cli.audit_dataset --manifest {SWAP} --root-dir {ROOT} --out {OUT}/audit_swap.json

## 7. Train every variant on the swaps

Same presets, seeds and selection rule as before: checkpoints and the best model are chosen on
**validation** macro-F1; test is reported once. All presets share `SCHEDULE` (set in the first cell);
the default is the longer, lower-learning-rate schedule of run 2.

In [ ]:
!python -m av_contradiction.cli.run_ablations {PATHS} --set experiment_name={EXPERIMENT} --set data.manifest={SWAP} {SCHEDULE} --seeds {SEEDS}

## 8. Extra tests

* the best swap-trained model on the **hard** swaps
* the released AVID model (`models/avid/`, trained on edited audio) on both swap test sets: does it
  detect natural mismatches at all?

In [ ]:
BEST = f"{OUT}/reports/{EXPERIMENT}/best_model/best.pt"
RELEASED = f"{CODE}/models/avid/best.pt"
for name, ckpt, manifest in [("swap_model_on_hard", BEST, HARD), ("avid_model_on_swap", RELEASED, SWAP),
                             ("avid_model_on_hard", RELEASED, HARD)]:
    print("==", name)
    !python -m av_contradiction.cli.evaluate --checkpoint {ckpt} --split test --no-calibration-file --output-dir {OUT}/reports         --set experiment_name=extra/{name} --set data.manifest={manifest} --set data.feature_cache_dir={OUT}/cache

## 9. Results

In [ ]:
import json

from IPython.display import Image, Markdown, display

rep = f"{OUT}/reports/{EXPERIMENT}"
display(Markdown(open(f"{rep}/ablation_summary.md", encoding="utf-8").read()))
display(Image(f"{rep}/ablation_macro_f1.png"))
print(json.dumps(json.load(open(f"{rep}/best_model/best_model.json")), indent=2))
for name in ("swap_model_on_hard", "avid_model_on_swap", "avid_model_on_hard"):
    m = json.load(open(f"{OUT}/reports/extra/{name}/metrics_test.json"))["metrics"]
    print(f"{name:20s} macro-F1 {m['macro_f1']:.3f}  accuracy {m['accuracy']:.3f}  ROC-AUC {m['roc_auc']:.3f}")

## 10. Download the results

`swap_results.zip` (in the *Output* tab) holds the summary tables, the swap manifests and their
reports, the extra test metrics and the best swap-trained model. Send the printed results above (or
the zip) back to update the README.

In [ ]:
RES = f"{OUT}/swap_results"
shutil.rmtree(RES, ignore_errors=True)
shutil.copytree(f"{OUT}/reports/{EXPERIMENT}", RES, ignore=shutil.ignore_patterns("seed*", "*.csv"))
shutil.copytree(f"{OUT}/reports/extra", f"{RES}/extra", ignore=shutil.ignore_patterns("*.csv", "*.png"))
for f in (SWAP, HARD, f"{OUT}/avid_swap_report.json", f"{OUT}/avid_swap_hard_report.json"):
    shutil.copy2(f, RES)
print(shutil.make_archive(f"{OUT}/swap_results", "zip", RES))